# Multi-Agent Travel Planner — Microsoft Agent Framework

The same travel planner as `travel_planner_langgraph_(1) (2).ipynb`, with current Microsoft Agent Framework APIs.

Manager → selected flight/hotel/activity workers **concurrently** → synthesis → critic → revision → final state.

The same travel data, tool function bodies, permissions, agent instructions, model prompts, worker selection and output headings are retained. Executor steps use direct state reads/writes, following the ticket-solver teaching pattern. Microsoft executors use native workflow shared state. Only scheduling signals travel between executors.

### Run in Google Colab

1. Connect to a standard CPU runtime (Python 3.10+). No GPU is required.
2. In Secrets, use `OPENAI_API_KEY` and enable **Notebook access for this notebook**.
3. Run the install cell. If Colab requests a restart, restart and continue at Runtime and credentials. Restart first if framework packages were imported before installing.
4. Run the remaining cells in order. The demo uses actual OpenAI calls. The final cell accepts your own trip request.

To change the model or key, rerun setup and all agent/workflow definitions. The OpenAI key must have API billing/quota and access to the selected model. Local Python 3.12/3.13 and intercepted-provider checks do not establish hosted Colab or paid-call success.


## Google Colab setup

Use a fresh **Python 3.12 or 3.13 CPU runtime** for each notebook. In the Secrets sidebar, add `OPENAI_API_KEY` and enable **Notebook access** for this notebook. Run cells from top to bottom. If installation requests a restart, restart the session and continue with the imports/credentials cell (do not reinstall repeatedly). No Drive mount or local project files are needed.

The default model is `gpt-4o-mini`. API billing/quota and model access are required; a ChatGPT subscription does not supply API quota. Keys are never printed. Package conflicts can occur if multiple framework notebooks share one runtime—start a fresh session when switching frameworks.


In [ ]:
import sys
if sys.version_info < (3, 11):
    raise RuntimeError("Use Python 3.11 or newer. In Colab choose a current Python runtime.")
print("Runtime Python:", sys.version.split()[0])
%pip -q install "agent-framework-core==1.21.0" "agent-framework-openai==1.16.0"
print("Install complete. If Colab requests a restart, restart and continue at Runtime and credentials.")

## 1. Runtime and credentials

The notebook uses your Colab secret automatically. If Colab cannot read it, the cell explains the missing-secret, permission or unavailable-UI case without printing exception details or key values. It then tries an existing `OPENAI_API_KEY` environment variable and finally a hidden prompt. Local Jupyter users can set the environment variable before launching Jupyter.

The model stays `gpt-4o-mini`, matching the source. An API key must also have API billing/quota and access to the selected model. Change `MODEL` explicitly only if your account requires another model. This setup checks that a key exists, not whether the server accepts it.

Jupyter and Colab support top-level `await`. Do not wrap the calls below in `asyncio.run()`.

In [ ]:
import os, asyncio
from typing import Never
from agent_framework import Agent, WorkflowBuilder, WorkflowContext, executor, tool
from agent_framework.openai import OpenAIChatCompletionClient
from importlib.metadata import version

MODEL = "gpt-4o-mini"

def read_openai_key():
    # In Colab, always use this notebook's enabled secret.
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None
    if userdata is not None:
        try:
            key = userdata.get("OPENAI_API_KEY")
        except Exception:
            raise RuntimeError(
                "In Colab Secrets, add OPENAI_API_KEY and enable Notebook access, "
                "then rerun this cell."
            ) from None
        if not isinstance(key, str) or not key.strip():
            raise RuntimeError("The OPENAI_API_KEY Colab secret is empty.")
        return key.strip()

    # Local Python fallback; never overwrite an existing environment key.
    try:
        from dotenv import load_dotenv, find_dotenv
        load_dotenv(find_dotenv(usecwd=True), override=False)
    except ImportError:
        pass
    key = os.environ.get("OPENAI_API_KEY", "").strip()
    if not key:
        from getpass import getpass
        key = getpass("Enter OPENAI_API_KEY (hidden): ").strip()
    if not key:
        raise ValueError("OPENAI_API_KEY cannot be empty.")
    return key

os.environ["OPENAI_API_KEY"] = read_openai_key()
print("Model:", MODEL)
print("Framework versions:", version("agent-framework-core"), version("agent-framework-openai"))

## 2. The same travel data
Static classroom fixtures, copied verbatim. Rates, weather and visa tips are sample data, not current travel advice or live booking inventory.

In [ ]:
# ---------- FLIGHTS ----------
FLIGHTS = [
    {"id": "SQ-421", "from": "Mumbai", "to": "Singapore", "airline": "Singapore Airlines",
     "depart": "11:45", "arrive": "19:50", "stops": 0, "price_inr": 28500, "red_eye": False},
    {"id": "AI-342", "from": "Mumbai", "to": "Singapore", "airline": "Air India",
     "depart": "23:15", "arrive": "07:20", "stops": 0, "price_inr": 23600, "red_eye": True},
    {"id": "6E-1071", "from": "Mumbai", "to": "Singapore", "airline": "IndiGo",
     "depart": "08:30", "arrive": "16:40", "stops": 0, "price_inr": 21900, "red_eye": False},
    {"id": "MH-199", "from": "Mumbai", "to": "Singapore", "airline": "Malaysia Airlines",
     "depart": "14:20", "arrive": "23:55", "stops": 1, "price_inr": 19400, "red_eye": False},
    {"id": "SQ-423", "from": "Delhi", "to": "Singapore", "airline": "Singapore Airlines",
     "depart": "10:10", "arrive": "19:05", "stops": 0, "price_inr": 31200, "red_eye": False},
    {"id": "6E-1085", "from": "Bangalore", "to": "Singapore", "airline": "IndiGo",
     "depart": "09:05", "arrive": "16:20", "stops": 0, "price_inr": 20800, "red_eye": False},
]

# ---------- HOTELS ----------
HOTELS = [
    {"name": "Hotel Boss", "city": "Singapore", "area": "Lavender",
     "price_per_night_inr": 7200, "rating": 4.0, "style": "budget"},
    {"name": "Village Hotel Bugis", "city": "Singapore", "area": "Bugis",
     "price_per_night_inr": 9800, "rating": 4.3, "style": "mid"},
    {"name": "Hotel G Singapore", "city": "Singapore", "area": "Bencoolen",
     "price_per_night_inr": 8600, "rating": 4.2, "style": "mid"},
    {"name": "Marina Bay Sands", "city": "Singapore", "area": "Marina Bay",
     "price_per_night_inr": 34000, "rating": 4.7, "style": "luxury"},
    {"name": "Hotel Mono", "city": "Singapore", "area": "Chinatown",
     "price_per_night_inr": 10500, "rating": 4.4, "style": "mid"},
    {"name": "The Sultan", "city": "Singapore", "area": "Kampong Glam",
     "price_per_night_inr": 11200, "rating": 4.5, "style": "mid"},
]

# ---------- THINGS TO DO ----------
ACTIVITIES = [
    {"name": "Gardens by the Bay", "city": "Singapore", "interest": "city views", "hours": 3, "area": "Marina Bay"},
    {"name": "Marina Bay Sands SkyPark", "city": "Singapore", "interest": "city views", "hours": 2, "area": "Marina Bay"},
    {"name": "Singapore Flyer", "city": "Singapore", "interest": "city views", "hours": 2, "area": "Marina Bay"},
    {"name": "Chinatown Heritage Centre", "city": "Singapore", "interest": "culture", "hours": 2, "area": "Chinatown"},
    {"name": "Sultan Mosque walk", "city": "Singapore", "interest": "culture", "hours": 2, "area": "Kampong Glam"},
    {"name": "Little India walking tour", "city": "Singapore", "interest": "culture", "hours": 3, "area": "Little India"},
    {"name": "Hawker food trail", "city": "Singapore", "interest": "food", "hours": 3, "area": "Chinatown"},
    {"name": "Sentosa beach day", "city": "Singapore", "interest": "relax", "hours": 5, "area": "Sentosa"},
]

# ---------- FOOD ----------
RESTAURANTS = [
    {"name": "Maxwell Food Centre", "city": "Singapore", "cuisine": "local", "cost_inr": 500, "area": "Chinatown"},
    {"name": "Lau Pa Sat", "city": "Singapore", "cuisine": "local", "cost_inr": 600, "area": "Downtown"},
    {"name": "Komala Vilas", "city": "Singapore", "cuisine": "indian veg", "cost_inr": 700, "area": "Little India"},
    {"name": "Zam Zam", "city": "Singapore", "cuisine": "halal", "cost_inr": 800, "area": "Kampong Glam"},
    {"name": "Newton Food Centre", "city": "Singapore", "cuisine": "seafood", "cost_inr": 1600, "area": "Newton"},
    {"name": "Tiong Bahru Market", "city": "Singapore", "cuisine": "local", "cost_inr": 450, "area": "Tiong Bahru"},
]

# ---------- WEATHER ----------
WEATHER = {
    "singapore": "Hot and humid all year, 26-32 C. Short heavy rain most afternoons. Carry an umbrella.",
    "bangkok":   "Hot, 28-35 C. Heavy rain June to October.",
    "dubai":     "Very hot April to September, 35-45 C. Pleasant November to March.",
}

# ---------- LOCAL TIPS ----------
TIPS = {
    "singapore": "Indian passport holders need a visa. MRT covers the whole city. Buy an EZ-Link card. Tap water is safe. No chewing gum.",
    "bangkok":   "Visa on arrival for Indians. Use BTS Skytrain. Always agree the taxi fare first.",
    "dubai":     "Visa needed in advance. Metro is clean and cheap. Dress modestly in public places.",
}

# ---------- EXCHANGE RATES (1 unit = ? INR) ----------
RATES = {"SGD": 65.0, "USD": 88.0, "THB": 2.5, "AED": 24.0, "INR": 1.0}

print("Data loaded:", len(FLIGHTS), "flights,", len(HOTELS), "hotels,",
      len(ACTIVITIES), "activities,", len(RESTAURANTS), "restaurants.")

## 3. The same tools and permissions
Microsoft `@tool` replaces LangChain `@tool`. The function bodies and descriptions stay identical. Microsoft exposes `FunctionTool`; use `.func(...)` for a direct Python check. `retriver` remains unregistered, as in the source.

In [ ]:



@tool
def search_flights(origin: str, destination: str, avoid_red_eye: bool = False,
                   max_price_inr: int = 999999) -> list:
    """Find flights between two cities. Set avoid_red_eye to True to remove overnight flights."""
    # FLIGHTS = await("https://jsonplaceholder.typicode.com/flights")
      # FLIGHTS = select * from flight where time = 12-2-2026
    found = [f for f in FLIGHTS
             if f["from"].lower() == origin.lower()
             and f["to"].lower() == destination.lower()
             and f["price_inr"] <= max_price_inr]
    if avoid_red_eye:
        found = [f for f in found if not f["red_eye"]]
    return sorted(found, key=lambda f: f["price_inr"])


@tool
def search_hotels(city: str, max_price_per_night_inr: int = 999999, style: str = "any") -> list:
    """Find hotels in a city. Style can be budget, mid, luxury or any."""
    found = [h for h in HOTELS
             if h["city"].lower() == city.lower()
             and h["price_per_night_inr"] <= max_price_per_night_inr]
    if style != "any":
        found = [h for h in found if h["style"] == style.lower()]
    return sorted(found, key=lambda h: -h["rating"])


@tool
def search_activities(city: str, interest: str = "any") -> list:
    """Find things to do in a city. Interest can be food, culture, city views, relax or any."""
    found = [a for a in ACTIVITIES if a["city"].lower() == city.lower()]
    if interest != "any":
        found = [a for a in found if interest.lower() in a["interest"]]
    return found


@tool
def search_restaurants(city: str, cuisine: str = "any") -> list:
    """Find places to eat in a city. Cuisine can be local, indian veg, halal, seafood or any."""
    found = [r for r in RESTAURANTS if r["city"].lower() == city.lower()]
    if cuisine != "any":
        found = [r for r in found if cuisine.lower() in r["cuisine"]]
    return sorted(found, key=lambda r: r["cost_inr"])


@tool
def retriver(city: str, cuisine: str = "any") -> list:
    """he will find the right content from document we have"""
    found = [r for r in RESTAURANTS if r["city"].lower() == city.lower()]
    if cuisine != "any":
        found = [r for r in found if cuisine.lower() in r["cuisine"]]
    return sorted(found, key=lambda r: r["cost_inr"])



@tool
def get_weather(city: str) -> str:
    """Get the usual weather for a city, so the plan can allow for rain or heat."""
    return WEATHER.get(city.lower(), "No weather information for this city.")


@tool
def get_travel_tips(city: str) -> str:
    """Get visa, transport and local rules for a city."""
    return TIPS.get(city.lower(), "No tips available for this city.")


@tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert money between currencies, for example SGD to INR."""
    if from_currency.upper() not in RATES or to_currency.upper() not in RATES:
        return "Currency not supported."
    in_inr = amount * RATES[from_currency.upper()]
    result = in_inr / RATES[to_currency.upper()]
    return f"{amount} {from_currency.upper()} = {round(result, 2)} {to_currency.upper()}"


print("8 tools defined; 7 distinct tools registered below.")

In [ ]:
search_flights.func(origin="Mumbai", destination="Singapore", avoid_red_eye=True)

In [ ]:
TOOL_REGISTRY = {
    "flight_agent":   [search_flights, convert_currency],
    "hotel_agent":    [search_hotels, convert_currency, get_travel_tips],
    "activity_agent": [search_activities, search_restaurants, get_weather],
    "critic_agent":   [],   # the critic only reads the plan, it does not search
}

# Show the registry as a small table
for agent_name, tools in TOOL_REGISTRY.items():
    tool_names = [t.name for t in tools] or ["(no tools)"]
    print(f"{agent_name:16} -> {', '.join(tool_names)}")

## 4. Create the OpenAI agents

Each specialist receives a separate Chat Completions provider client, the original instructions and its tool registry. `await agent.run(question)` performs the model/tool loop. Calls omit a shared AgentSession, matching the original fresh-conversation behavior.

The model defaults to `gpt-4o-mini`, as in the original. No seed or extra sampling settings are introduced; the original does not set them.


In [ ]:
def make_client():
    return OpenAIChatCompletionClient(
        model=MODEL,
        api_key=os.environ["OPENAI_API_KEY"],
    )

In [ ]:
async def ask(agent, question: str) -> str:
    """Fresh call, no shared AgentSession: same stateless semantics as source."""
    return (await agent.run(question)).text

manager_agent = Agent(client=make_client(), tools=[], instructions='You are the manager of a travel planning team. You never search for travel data yourself. You read the request, decide who should work, and write the final plan. Be short and clear. Never invent a flight, hotel or place that a worker did not give you.')
flight_agent = Agent(client=make_client(), tools=TOOL_REGISTRY['flight_agent'], instructions='You are the flight specialist. Use your tools to find real options. Never suggest a flight that did not come from a tool. Answer in exactly this shape:\nTOP 3 OPTIONS: a numbered list with airline, flight id, times and price\nRECOMMENDED: one option and one line saying why\nTRADE-OFFS: what the traveller gives up by taking it\nASSUMPTIONS: anything you had to guess')
hotel_agent = Agent(client=make_client(), tools=TOOL_REGISTRY['hotel_agent'], instructions='You are the hotel specialist. Use your tools to find real options. Never suggest a hotel that did not come from a tool. Always check the total cost for the whole stay, not only the price per night. Answer in exactly this shape:\nTOP 3 OPTIONS: a numbered list with name, area, rating, price per night, total for the stay\nRECOMMENDED: one option and one line saying why\nTRADE-OFFS: what the traveller gives up by taking it\nASSUMPTIONS: anything you had to guess')
activity_agent = Agent(client=make_client(), tools=TOOL_REGISTRY['activity_agent'], instructions='You are the activities specialist. Use your tools to find real places. Never suggest a place that did not come from a tool. Do not overload a day. Two or three activities plus one meal is enough. Answer in exactly this shape:\nDAY BY DAY: for each day, morning, afternoon, evening and where to eat\nWEATHER NOTE: one line\nASSUMPTIONS: anything you had to guess')
critic_agent = Agent(client=make_client(), tools=TOOL_REGISTRY['critic_agent'], instructions="You are a strict reviewer of travel plans. You do not rewrite the plan. You list problems only. Check four things: budget respected, days not overloaded, the traveller's stated interests actually appear, and guesses are stated honestly. Answer as a numbered list of specific fixes. Never write vague comments like 'make it better'. If the plan is fine, say NO ISSUES FOUND.")
print('5 agents ready: manager, flight, hotel, activity, critic.')

## 5. State fields and direct executor steps

`TravelState` describes the eventual result fields for Python typing. It does not create the Microsoft state store or register a schema with the builder. The workflow runtime owns that store and supplies a context to each executor. Initially only the request arrives as an input message. The manager writes it into state and adds requirements and worker names. Later steps create the remaining fields.

The functions below follow the ticket solver pattern: `@executor` above each step, direct `ctx.get_state` / `ctx.set_state` calls, and a separate graph builder. Every model prompt and the selection/review policy remain the same.

This store differs from LangGraph state: reads are copied, writes commit at superstep boundaries, and conflicts do not use LangGraph reducers. Workers return reports before this executor stores each one under its own key. Unselected report fields stay absent. No checkpoint or shared AgentSession is enabled.


In [ ]:
from typing import TypedDict, List, cast
from typing_extensions import NotRequired


class TravelState(TypedDict):
    request: str
    requirements: NotRequired[str]
    chosen_workers: NotRequired[List[str]]
    flight_report: NotRequired[str]
    hotel_report: NotRequired[str]
    activity_report: NotRequired[str]
    draft_plan: NotRequired[str]
    critique: NotRequired[str]
    final_plan: NotRequired[str]


print("TravelState schema defined: request required; derived fields optional.")

In [ ]:
@executor(id='manager')
async def manager_step(request: str, ctx: WorkflowContext[str]):
    """Read the request. Write down the requirements. Decide who should work."""
    ctx.set_state('request', request)
    requirements = await ask(manager_agent, f"\nRead this travel request and list what you understood.\n\nREQUEST: {ctx.get_state('request')}\n\nList these lines only:\nFROM, TO, DAYS, TRAVELLERS, BUDGET, INTERESTS, HARD RULES, MISSING INFO\n")
    choice = await ask(manager_agent, f"""
Here are the requirements:
{requirements}

Which specialists are needed? Choose from: flight_agent, hotel_agent, activity_agent.
Reply with the names separated by commas and nothing else.
""")
    valid = ['flight_agent', 'hotel_agent', 'activity_agent']
    workers = [name for name in valid if name in choice.lower()]
    if not workers:
        workers = valid
    print('MANAGER understood the request.')
    print('MANAGER picked:', ', '.join(workers))
    ctx.set_state('requirements', requirements)
    ctx.set_state('chosen_workers', workers)
    await ctx.send_message('continue')

@executor(id="selected_workers")
async def workers_step(signal: str, ctx: WorkflowContext[str]) -> None:
    requirements = ctx.get_state("requirements")
    chosen_workers = ctx.get_state("chosen_workers")

    # Each entry contains the agent, its instruction and its state field.
    jobs = {
        "flight_agent": (flight_agent, "Find flights for this trip.", "flight_report"),
        "hotel_agent": (hotel_agent, "Find hotels for this trip.", "hotel_report"),
        "activity_agent": (activity_agent, "Plan the days for this trip.", "activity_report"),
    }
    selected_jobs = [jobs[name] for name in chosen_workers]

    # Wait for ALL selected specialists before storing reports and continuing.
    reports = await asyncio.gather(*(
        ask(agent, f"{instruction}\n\n{requirements}")
        for agent, instruction, key in selected_jobs
    ))
    for (_, _, key), report in zip(selected_jobs, reports):
        ctx.set_state(key, report)

    await ctx.send_message("continue")


@executor(id='synthesis')
async def synthesis_step(signal: str, ctx: WorkflowContext[str]):
    """The manager joins the three reports into one travel plan."""
    draft = await ask(manager_agent, f"\nWrite the first version of the travel plan using ONLY the reports below.\n\nREQUIREMENTS:\n{ctx.get_state('requirements')}\n\nFLIGHT REPORT:\n{ctx.get_state('flight_report', 'not requested')}\n\nHOTEL REPORT:\n{ctx.get_state('hotel_report', 'not requested')}\n\nACTIVITY REPORT:\n{ctx.get_state('activity_report', 'not requested')}\n\nUse these headings:\nCHOSEN FLIGHT, CHOSEN HOTEL, DAY BY DAY PLAN, TOTAL COST, WHY THESE CHOICES, ASSUMPTIONS\n")
    print('MANAGER wrote the first plan.')
    ctx.set_state('draft_plan', draft)
    await ctx.send_message('continue')

@executor(id='critic')
async def critic_step(signal: str, ctx: WorkflowContext[str]):
    """The critic looks for problems. It does not fix them."""
    issues = await ask(critic_agent, f"\nReview this plan against the requirements.\n\nREQUIREMENTS:\n{ctx.get_state('requirements')}\n\nPLAN:\n{ctx.get_state('draft_plan')}\n")
    print('CRITIC finished the review.')
    ctx.set_state('critique', issues)
    await ctx.send_message('continue')

@executor(id='revision')
async def revision_step(signal: str, ctx: WorkflowContext[Never, dict]):
    """The manager fixes only what the critic listed. Everything else stays as it was."""
    if 'NO ISSUES FOUND' in ctx.get_state('critique').upper():
        print('No changes needed.')
        ctx.set_state('final_plan', ctx.get_state('draft_plan'))
    else:
        fixed = await ask(manager_agent, f"\nFix ONLY the problems listed below. Do not rewrite the parts that are fine.\n\nPLAN:\n{ctx.get_state('draft_plan')}\n\nPROBLEMS TO FIX:\n{ctx.get_state('critique')}\n\nReturn the full corrected plan, then a short section called WHAT CHANGED.\n")
        print('MANAGER fixed the plan.')
        ctx.set_state('final_plan', fixed)
    result = {'request': ctx.get_state('request'), 'requirements': ctx.get_state('requirements'), 'chosen_workers': ctx.get_state('chosen_workers'), 'draft_plan': ctx.get_state('draft_plan'), 'critique': ctx.get_state('critique'), 'final_plan': ctx.get_state('final_plan')}
    for key in ['flight_report', 'hotel_report', 'activity_report']:
        report = ctx.get_state(key)
        if report is not None:
            result[key] = report
    await ctx.yield_output(result)


## 6. Connect the workflow

Same pattern as the ticket solver: decorated executor functions above, graph wiring below.

Each executor reads with `ctx.get_state` and writes with `ctx.set_state`. There are no snapshot/update adapter functions. The manager stores the initial request, the workers store their reports, synthesis stores the draft, the critic stores its review, and revision yields the final result.

The selected specialists run concurrently inside one worker executor. `asyncio.gather` waits for all of them before synthesis can start. State writes do not schedule another step: `ctx.send_message("continue")` does. The terminal executor uses `ctx.yield_output(result)`.


In [ ]:
def create_workflow():
    return (
        WorkflowBuilder(start_executor=manager_step, output_from=[revision_step])
        .add_edge(manager_step, workers_step)
        .add_edge(workers_step, synthesis_step)
        .add_edge(synthesis_step, critic_step)
        .add_edge(critic_step, revision_step)
        .build()
    )


async def run_travel_state(request: str) -> TravelState:
    run_result = await create_workflow().run(request)
    outputs = run_result.get_outputs()
    if len(outputs) != 1:
        raise RuntimeError(f"Expected one final state, got {len(outputs)}")
    return cast(TravelState, outputs[0])


## 7. Same printed trace and demo
Use top-level `await` in Jupyter. The original `return result['fina']` raises `KeyError`, and its next validation cell expects a dictionary. This version returns the complete result dictionary; `result["final_plan"]` is the final text.

In [ ]:
async def plan_trip(request: str, show_details: bool = True):
    """Run the full agent team on a travel request."""
    print("=" * 70)
    print("REQUEST:", request)
    print("=" * 70)

    result = await run_travel_state(request)

    if show_details:
        print("\n" + "-" * 70)
        print("WHAT THE MANAGER UNDERSTOOD")
        print("-" * 70)
        print(result["requirements"])

        for key, title in [("flight_report", "FLIGHT AGENT"),
                           ("hotel_report", "HOTEL AGENT"),
                           ("activity_report", "ACTIVITY AGENT")]:
            if result.get(key):
                print("\n" + "-" * 70)
                print(title)
                print("-" * 70)
                print(result[key])

        print("\n" + "-" * 70)
        print("FIRST DRAFT")
        print("-" * 70)
        print(result["draft_plan"])

        print("\n" + "-" * 70)
        print("WHAT THE CRITIC FOUND")
        print("-" * 70)
        print(result["critique"])

    print("\n" + "=" * 70)
    print("FINAL TRAVEL PLAN")
    print("=" * 70)
    print(result["final_plan"])
    return result


print("Ready to run.")

In [ ]:
demo_request = """Plan a 4-day Singapore trip from Mumbai for two people.
Mid-budget. Avoid red-eye flights.
Prefer food, city views and cultural sites.
Keep hotel budget under 45000 rupees."""

result = await plan_trip(demo_request)

## 8. Inspect grounding
These are the same source checks: they show known names present in text. They do **not** detect all invented names or validate arithmetic; inspect the reports and totals too.

In [ ]:
# Check that every flight id in the final plan really exists in our data
plan_text = result["final_plan"]

real_ids = [f["id"] for f in FLIGHTS]
found_ids = [fid for fid in real_ids if fid in plan_text]
print("Real flight ids used in the plan:", found_ids or "none")

real_hotels = [h["name"] for h in HOTELS]
found_hotels = [name for name in real_hotels if name in plan_text]
print("Real hotel names used in the plan:", found_hotels or "none")

print("\nIf both lines are empty, the plan was invented and the grounding check has failed.")

## 9. Your trip

Enter another travel request. The same live agent team runs with a fresh workflow state for each trip. No travel booking or external travel API is performed.


In [ ]:
your_request = input("Describe your trip: ")
result = await plan_trip(your_request)

## References and fidelity notes

Verified against `agent-framework-core==1.21.0` and `agent-framework-openai==1.16.0`:

- [Microsoft OpenAI provider and current Agent API](https://learn.microsoft.com/en-us/agent-framework/integrations/by-component/model-providers/openai)
- [WorkflowContext native state APIs](https://github.com/microsoft/agent-framework/blob/main/python/packages/core/agent_framework/_workflows/_workflow_context.py)
- [Native State buffering and commit semantics](https://github.com/microsoft/agent-framework/blob/main/python/packages/core/agent_framework/_workflows/_state.py)
- [Google Colab Secrets API](https://github.com/googlecolab/colabtools/blob/main/google/colab/userdata.py)
- [Colab runtime versions](https://research.google.com/colaboratory/runtime-version-faq.html)

The original is unchanged. Its `return result['fina']` typo is corrected here to return the full result dictionary, as the following grounding cell expects. Eight tools are defined and seven distinct tools are registered, retaining the unused `retriver` function.

Both frameworks have workflow state, with different merge/conflict semantics. Derived fields start absent. This notebook does not enable checkpointing or shared agent chat history.

Developer validation lives separately in `../scripts/validate_microsoft_travel_planner.py`. It intercepts provider HTTP responses to check the actual live code without paid calls. Hosted Colab, real authentication, billing/quota, generated-plan quality and real-world travel accuracy remain untested.
